# 03 – Feature Engineering Badgeuse
## HumanForYou – Attrition ML

Objectif : calculer **une seule variable temporelle** à partir des données badgeuse :
`avg_work_hours` = moyenne des heures travaillées par jour (out_time − in_time).

## 1. Import et chargement

In [6]:
import pandas as pd
import numpy as np
import os

RAW_DIR = os.path.join('..', 'data', 'raw')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')

df_base = pd.read_csv(os.path.join(PROCESSED_DIR, 'attrition_merged_base.csv'))
in_time = pd.read_csv(os.path.join(RAW_DIR, 'in_out_time', 'in_time.csv'))
out_time = pd.read_csv(os.path.join(RAW_DIR, 'in_out_time', 'out_time.csv'))

in_time.rename(columns={in_time.columns[0]: 'EmployeeID'}, inplace=True)
out_time.rename(columns={out_time.columns[0]: 'EmployeeID'}, inplace=True)

print(f'df_base  : {df_base.shape}')
print(f'in_time  : {in_time.shape}')
print(f'out_time : {out_time.shape}')

df_base  : (4410, 26)
in_time  : (4410, 262)
out_time : (4410, 262)


## 2. Conversion en datetime et calcul des durées

In [7]:
day_cols = [c for c in in_time.columns if c != 'EmployeeID']
print(f'Nombre de jours dans les données badgeuse : {len(day_cols)}')

# Convertir chaque colonne en datetime
in_dt = in_time[day_cols].apply(pd.to_datetime, errors='coerce')
out_dt = out_time[day_cols].apply(pd.to_datetime, errors='coerce')

# Durée en heures pour chaque jour
duration_hours = (out_dt - in_dt).apply(lambda col: col.dt.total_seconds() / 3600)

# Garder uniquement les durées positives
duration_hours[duration_hours <= 0] = np.nan

print(f'Exemple durées (ligne 0) : {duration_hours.iloc[0].dropna().head().round(2).tolist()}')

Nombre de jours dans les données badgeuse : 261
Exemple durées (ligne 0) : [7.21, 7.19, 7.41, 7.01, 7.29]


## 3. Calcul de avg_work_hours

In [8]:
avg_work_hours = duration_hours.mean(axis=1)

print(f'avg_work_hours :')
print(f'  count  : {avg_work_hours.notna().sum()}')
print(f'  mean   : {avg_work_hours.mean():.2f} h')
print(f'  std    : {avg_work_hours.std():.2f} h')
print(f'  min    : {avg_work_hours.min():.2f} h')
print(f'  max    : {avg_work_hours.max():.2f} h')
print(f'  NaN    : {avg_work_hours.isna().sum()}')

avg_work_hours :
  count  : 4410
  mean   : 7.70 h
  std    : 1.34 h
  min    : 5.95 h
  max    : 11.03 h
  NaN    : 0


## 4. Imputation des NaN et merge

In [9]:
# Imputer les rares NaN par la médiane globale
median_hours = avg_work_hours.median()
avg_work_hours = avg_work_hours.fillna(median_hours)
print(f'Médiane utilisée pour imputation : {median_hours:.2f} h')
print(f'NaN restants : {avg_work_hours.isna().sum()}')

# Créer le DataFrame de features
features = pd.DataFrame({
    'EmployeeID': in_time['EmployeeID'],
    'avg_work_hours': avg_work_hours
})

# Merge avec le dataset principal
df = df_base.merge(features, on='EmployeeID', how='left')

# Imputer si NaN après le merge left
if df['avg_work_hours'].isna().sum() > 0:
    df['avg_work_hours'] = df['avg_work_hours'].fillna(median_hours)

print(f'\ndf apres merge : {df.shape}')
print(f'NaN total : {df.isna().sum().sum()}')

Médiane utilisée pour imputation : 7.41 h
NaN restants : 0

df apres merge : (4410, 27)
NaN total : 0


## 5. Export et validation

In [10]:
assert df.shape[0] == 4410, f'Nombre de lignes inattendu : {df.shape[0]}'
assert df.isna().sum().sum() == 0, f'NaN restants : {df.isna().sum().sum()}'
assert 'avg_work_hours' in df.columns, 'avg_work_hours manquant'

output_path = os.path.join(PROCESSED_DIR, 'attrition_with_avg_hours.csv')
df.to_csv(output_path, index=False)

print(f'Shape   : {df.shape}')
print(f'NaN     : {df.isna().sum().sum()}')
print(f'Export  : {output_path}')
print(f'\navg_work_hours : mean={df["avg_work_hours"].mean():.2f}, std={df["avg_work_hours"].std():.2f}')

Shape   : (4410, 27)
NaN     : 0
Export  : ..\data\processed\attrition_with_avg_hours.csv

avg_work_hours : mean=7.70, std=1.34
